In [3]:

# PHASE 0 - SETUP
# Connect Google Drive and extract the CMU starter project.

from google.colab import drive, files
from pathlib import Path
import os
import zipfile

# Step 1: Connect Google Drive.
drive.mount("/content/drive")

# Step 2: Define the project folder.
PROJECT = Path(
    "/content/drive/MyDrive/InsightForge_CMU/insightforge"
)

PROJECT.mkdir(parents=True, exist_ok=True)

# Step 3: Check whether we already extracted the starter.
essential_files = [
    PROJECT / "research_agent/agent.py",
    PROJECT / "requirements.txt",
    PROJECT / "tests/test_agent.py",
]

if all(path.exists() for path in essential_files):
    print("Starter already exists. No files overwritten.")

elif any(path.exists() for path in essential_files):
    raise RuntimeError(
        "Some starter files already exist. "
        "Check your folder before extracting again."
    )

else:
    print("Upload CMU_research_agent_starter.zip")
    uploaded = files.upload()

    if list(uploaded.keys()) != [
        "CMU_research_agent_starter.zip"
    ]:
        raise ValueError("Please upload only the CMU starter ZIP.")

    with zipfile.ZipFile(
        "CMU_research_agent_starter.zip"
    ) as archive:
        archive.extractall(PROJECT)

    print("Starter extracted successfully!")

# Step 4: Open our project directory.
os.chdir(PROJECT)

# Step 5: Verify important files.
for path in essential_files:
    status = "OK" if path.exists() else "MISSING"
    print(path.relative_to(PROJECT), status)

print("Working directory:", PROJECT)


Mounted at /content/drive
Starter already exists. No files overwritten.
research_agent/agent.py OK
requirements.txt OK
tests/test_agent.py OK
Working directory: /content/drive/MyDrive/InsightForge_CMU/insightforge


In [4]:

# PHASE 0 - INSTALL DEPENDENCIES AND RUN TESTS

# Change to the project directory.
%cd /content/drive/MyDrive/InsightForge_CMU/insightforge

# Install the libraries required by the starter.
%pip install -q -r requirements.txt

# Run automated tests.
!python -m pytest tests -q


/content
..                                                                       [100%]
2 passed in 6.03s



# InsightForge
## AI Research Assistant for Product and Technology Decisions

**CMU Capstone Category:** Research Assistant

### Problem Statement
InsightForge will help product managers and technical teams
research technology problems and compare alternative solutions.

Our primary example involves a hypothetical fintech company
evaluating AI-assisted production incident classification and routing.

### Project Goal
Build a research agent that collects evidence, evaluates
alternative solutions, identifies risks and produces a
source-supported decision brief.

The final decision remains with the human user.

### Starting Point
This project begins with the official CMU Research Agent
Starter Template.

### Phase 0: Baseline
The initial source code and tests come from the course starter.

Planned enhancements include:
- ReAct reasoning and tool selection
- Semantic retrieval using RAG
- Tree-of-Thought alternative evaluation
- Multi-agent coordination
- Safety guardrails and human escalation
- Evaluation and execution traces

Only publicly available and synthetic data will be used.


In [5]:

# PHASE 1A - OPEN EXISTING PROJECT

from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

PROJECT = Path(
    "/content/drive/MyDrive/InsightForge_CMU/insightforge"
)

# Make sure we're using our original CMU project.
assert (PROJECT / "research_agent/agent.py").exists(), (
    "CMU starter not found. Check your Drive path."
)

os.chdir(PROJECT)

print("Working in:", PROJECT)
print("CMU research agent found!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working in: /content/drive/MyDrive/InsightForge_CMU/insightforge
CMU research agent found!


Phase 1A — Evidence Collection and Research Tool
I am extending the CMU Research Agent Starter with a read-only research tool.

My first implementation uses synthetic incident-triage documents so that I can test retrieval, source tracking and research behavior without relying on live websites or confidential company information.

This version uses keyword matching. Semantic retrieval, LLM tool selection and reasoning loops will be introduced in subsequent phases.

In [6]:

# PHASE 1A - CREATE SYNTHETIC EVIDENCE

import json

records = [
    {
        "id": "SYN-001",
        "title": "High-severity routing approval policy",
        "source_type": "synthetic_policy",
        "viewpoint": "mandatory_constraint",
        "date": "2026-11-19",
        "hard_requirement": True,
        "text": (
            "In this hypothetical company, P0 and P1 high-severity "
            "incidents require human approval before any "
            "AI-recommended team routing is applied. The assistant "
            "may suggest a team but must not perform routing changes."
        )
    },
    {
        "id": "SYN-002",
        "title": "Manual incident triage workload",
        "source_type": "synthetic_interview",
        "viewpoint": "potential_benefit",
        "date": "2026-11-19",
        "hard_requirement": False,
        "text": (
            "In this fictional engineering scenario, repeated "
            "manual categorization and ticket routing are considered "
            "time-consuming. The company wants to test whether "
            "suggestions could reduce effort; no performance "
            "improvement has been measured."
        )
    },
    {
        "id": "SYN-003",
        "title": "Risk of incorrect incident classification",
        "source_type": "synthetic_risk_note",
        "viewpoint": "risk",
        "date": "2026-11-19",
        "hard_requirement": False,
        "text": (
            "Incorrectly classifying or routing an incident may "
            "delay response. A safe pilot should include uncertainty "
            "flags, a manual fallback, and a method for engineers "
            "to correct suggestions."
        )
    },
    {
        "id": "SYN-004",
        "title": "Option A: rule-based triage",
        "source_type": "synthetic_design_option",
        "viewpoint": "alternative",
        "date": "2026-11-19",
        "hard_requirement": False,
        "text": (
            "Rule-based incident classification uses explicit "
            "patterns and routing rules. It is easy to audit "
            "but may require maintenance when incident descriptions "
            "or team ownership change."
        )
    },
    {
        "id": "SYN-005",
        "title": "Option B: RAG-assisted classification",
        "source_type": "synthetic_design_option",
        "viewpoint": "alternative",
        "date": "2026-11-19",
        "hard_requirement": False,
        "text": (
            "RAG-assisted incident categorization retrieves relevant "
            "runbooks and resolved ticket examples to ground an LLM "
            "suggestion. Retrieval quality and up-to-date documents "
            "remain important constraints."
        )
    },
    {
        "id": "SYN-006",
        "title": "Option C: agentic incident investigation",
        "source_type": "synthetic_design_option",
        "viewpoint": "alternative",
        "date": "2026-11-19",
        "hard_requirement": False,
        "text": (
            "An agentic triage assistant could call read-only tools "
            "to inspect synthetic incident context and recommend "
            "a team. Tool permissions, latency, and human oversight "
            "need explicit limits."
        )
    },
    {
        "id": "SYN-007",
        "title": "New solutions and missing evidence",
        "source_type": "synthetic_evaluation_note",
        "viewpoint": "uncertainty",
        "date": "2026-11-19",
        "hard_requirement": False,
        "text": (
            "A newer incident automation approach may have fewer "
            "public studies. Missing evidence is not the same as "
            "negative evidence; compare required pilot tests "
            "and document uncertainty."
        )
    },
    {
        "id": "SYN-008",
        "title": "Data protection and tool restrictions",
        "source_type": "synthetic_policy",
        "viewpoint": "mandatory_constraint",
        "date": "2026-11-19",
        "hard_requirement": True,
        "text": (
            "The hypothetical organization requires synthetic "
            "or approved public test data. Research tools must "
            "be read-only and must not expose credentials, "
            "private incident records, or customer details."
        )
    }
]

DATA = PROJECT / "data/synthetic/research_notes.json"
DATA.parent.mkdir(parents=True, exist_ok=True)

DATA.write_text(
    json.dumps(records, indent=2),
    encoding="utf-8"
)

print("Saved research records:", len(records))
print("Location:", DATA)


Saved research records: 8
Location: /content/drive/MyDrive/InsightForge_CMU/insightforge/data/synthetic/research_notes.json


In [7]:

from pathlib import Path
import os

# Our permanent InsightForge project location
PROJECT = Path(
    "/content/drive/MyDrive/InsightForge_CMU/insightforge"
)

print("Current directory:", Path.cwd())
print("Project folder exists:", PROJECT.is_dir())
print(
    "Research agent folder exists:",
    (PROJECT / "research_agent").is_dir()
)
print(
    "Original agent.py exists:",
    (PROJECT / "research_agent/agent.py").is_file()
)


Current directory: /content/drive/MyDrive/InsightForge_CMU/insightforge
Project folder exists: True
Research agent folder exists: True
Original agent.py exists: True


In [9]:

# Navigate to our original project folder.

os.chdir(PROJECT)

print("Working directory changed to:", Path.cwd())

# Confirm the research_agent folder is accessible.
assert Path("research_agent/agent.py").exists()

print("Ready to create research_tools.py!")


Working directory changed to: /content/drive/MyDrive/InsightForge_CMU/insightforge
Ready to create research_tools.py!


In [11]:
import os
from pathlib import Path

PROJECT = Path(
    "/content/drive/MyDrive/InsightForge_CMU/insightforge"
)

os.chdir(PROJECT)

print("Current directory:", os.getcwd())
print("Research agent folder exists:", Path("research_agent").is_dir())

Current directory: /content/drive/MyDrive/InsightForge_CMU/insightforge
Research agent folder exists: True


In [8]:

%%writefile research_agent/research_tools.py
"""Read-only search over InsightForge's synthetic research notes."""

import json
import re
from pathlib import Path

from .models import SearchResult, SourceDocument


# Ignore common words when comparing research questions.
STOPWORDS = {
    "a", "an", "and", "are", "be", "can", "do", "for", "in",
    "is", "of", "on", "or", "our", "should", "the", "to",
    "we", "what", "with", "would"
}


def keywords(text: str) -> set[str]:
    """Extract useful lowercase words from text."""

    words = re.findall(r"[a-z0-9]+", text.lower())

    return {
        word for word in words
        if word not in STOPWORDS
    }


class SyntheticResearchClient:
    """Search synthetic evidence using the CMU web-client interface."""

    def __init__(self, data_path: str | Path):

        # Load the synthetic research collection.
        self.records = json.loads(
            Path(data_path).read_text(encoding="utf-8")
        )

        # Index documents by ID for retrieval.
        self.by_id = {
            record["id"]: record
            for record in self.records
        }

    def search(
        self,
        query: str,
        limit: int = 5
    ) -> list[SearchResult]:
        """Rank matching documents without accessing the internet."""

        query_words = keywords(query)

        # Handle empty questions or invalid limits.
        if not query_words or limit <= 0:
            return []

        ranked = []

        for record in self.records:

            title_words = keywords(record["title"])
            body_words = keywords(record["text"])

            # Matching title words count twice.
            score = 2 * len(query_words & title_words)
            score += len(query_words & body_words)

            if score > 0:
                ranked.append((score, record))

        # Rank by score, then use ID for reproducible ties.
        ranked.sort(
            key=lambda item: (-item[0], item[1]["id"])
        )

        # Convert matching records into CMU SearchResult objects.
        return [
            SearchResult(
                title=record["title"],
                url=f"synthetic://{record['id']}",
                snippet=record["text"][:180],
                query=query,
            )
            for _, record in ranked[:limit]
        ]

    def fetch_source(
        self,
        result: SearchResult,
        max_chars: int = 12000
    ) -> SourceDocument:
        """Retrieve a synthetic document by source ID."""

        source_id = result.url.removeprefix("synthetic://")
        record = self.by_id[source_id]

        return SourceDocument(
            title=record["title"],
            url=result.url,
            snippet=result.snippet,
            query=result.query,
            content=record["text"][:max_chars],
        )


Writing research_agent/research_tools.py


In [10]:

file_path = PROJECT / "research_agent/research_tools.py"

if file_path.is_file():
    print("SUCCESS! research_tools.py was created.")
    print("File size:", file_path.stat().st_size, "bytes")
else:
    print("ERROR: research_tools.py was not found.")


SUCCESS! research_tools.py was created.
File size: 2791 bytes


In [12]:
file_path = PROJECT / "research_agent/research_tools.py"

assert file_path.is_file(), "File was not created!"

print("SUCCESS! research_tools.py is saved.")

SUCCESS! research_tools.py is saved.


In [13]:

from pathlib import Path

DATA = PROJECT / "data/synthetic/research_notes.json"

print("Research data exists:", DATA.is_file())

if DATA.is_file():
    import json
    records = json.loads(DATA.read_text(encoding="utf-8"))
    print("Total research records:", len(records))


Research data exists: True
Total research records: 8


In [14]:

# PHASE 1A - TEST OUR RESEARCH TOOL

# Import the Python class we created.
from research_agent.research_tools import SyntheticResearchClient

# Load our synthetic research documents.
client = SyntheticResearchClient(DATA)

# Ask our first research question.
question = "high severity incident routing human approval"

# Retrieve up to 3 matching research documents.
results = client.search(question, limit=3)

print("QUESTION:", question)
print("TOTAL MATCHES:", len(results))

# Display the retrieved evidence.
for result in results:
    print("\nSource:", result.url)
    print("Title:", result.title)
    print("Excerpt:", result.snippet)


QUESTION: high severity incident routing human approval
TOTAL MATCHES: 3

Source: synthetic://SYN-001
Title: High-severity routing approval policy
Excerpt: In this hypothetical company, P0 and P1 high-severity incidents require human approval before any AI-recommended team routing is applied. The assistant may suggest a team but must 

Source: synthetic://SYN-003
Title: Risk of incorrect incident classification
Excerpt: Incorrectly classifying or routing an incident may delay response. A safe pilot should include uncertainty flags, a manual fallback, and a method for engineers to correct suggestio

Source: synthetic://SYN-006
Title: Option C: agentic incident investigation
Excerpt: An agentic triage assistant could call read-only tools to inspect synthetic incident context and recommend a team. Tool permissions, latency, and human oversight need explicit limi


In [15]:
results = client.search(question, limit=3)

In [16]:

# PHASE 1A - OPEN EXISTING PROJECT

from pathlib import Path
import os

PROJECT = Path(
    "/content/drive/MyDrive/InsightForge_CMU/insightforge"
)

# If Colab restarted, mount Google Drive first.
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

os.chdir(PROJECT)

DATA = PROJECT / "data/synthetic/research_notes.json"

# Verify the files we already created.
assert (PROJECT / "research_agent/agent.py").exists()
assert (PROJECT / "research_agent/research_tools.py").exists()
assert DATA.exists()

print("Project ready!")
print("Current directory:", Path.cwd())
print("Research data found:", DATA.name)


Project ready!
Current directory: /content/drive/MyDrive/InsightForge_CMU/insightforge
Research data found: research_notes.json


In [17]:

# PHASE 1A - INTEGRATE SYNTHETIC RESEARCH TOOL
# This uses the existing CMU ResearchAgent.

from research_agent.agent import (
    ResearchAgent,
    ResearchConfig,
    render_report
)

from research_agent.research_tools import (
    SyntheticResearchClient
)


# Create an offline report writer.
# This prevents external LLM API calls during testing.
class OfflineWriter:

    def write_report(self, topic, sources):
        return ""


# Step 1: Configure the original CMU agent.
config = ResearchConfig(
    max_queries=2,
    max_results_per_query=3,
    max_sources=3
)

agent = ResearchAgent(config)

# Step 2: Replace live web search with our research tool.
agent.web = SyntheticResearchClient(DATA)

# Step 3: Disable LLM report generation for this test.
agent.llm_writer = OfflineWriter()

# Step 4: Ask the research agent a question.
question = "high severity incident routing human approval"

report = agent.research(question)

# Step 5: Inspect the research results.
print("RESEARCH COMPLETE")
print("Question:", report.topic)
print("Number of sources:", len(report.references))

for source in report.references:
    print("\nSource ID:", source.url)
    print("Title:", source.title)
    print("Evidence:", source.content)

# Step 6: Display the completed research report.
print("\n--- INSIGHTFORGE RESEARCH REPORT ---\n")
print(render_report(report))


RESEARCH COMPLETE
Question: high severity incident routing human approval
Number of sources: 3

Source ID: synthetic://SYN-001
Title: High-severity routing approval policy
Evidence: In this hypothetical company, P0 and P1 high-severity incidents require human approval before any AI-recommended team routing is applied. The assistant may suggest a team but must not perform routing changes.

Source ID: synthetic://SYN-003
Title: Risk of incorrect incident classification
Evidence: Incorrectly classifying or routing an incident may delay response. A safe pilot should include uncertainty flags, a manual fallback, and a method for engineers to correct suggestions.

Source ID: synthetic://SYN-006
Title: Option C: agentic incident investigation
Evidence: An agentic triage assistant could call read-only tools to inspect synthetic incident context and recommend a team. Tool permissions, latency, and human oversight need explicit limits.

--- INSIGHTFORGE RESEARCH REPORT ---

# Research Report: hi

In [18]:

%%writefile tests/test_research_tools.py
"""Tests for InsightForge's synthetic research tool."""

from pathlib import Path

from research_agent.agent import ResearchAgent, ResearchConfig
from research_agent.research_tools import SyntheticResearchClient


# Locate our synthetic dataset relative to this test file.
DATA = (
    Path(__file__).resolve().parents[1]
    / "data/synthetic/research_notes.json"
)


class OfflineWriter:
    """Prevent LLM API calls during automated tests."""

    def write_report(self, topic, sources):
        return ""


def test_human_approval_policy_is_found():
    """The tool should retrieve the required policy."""

    client = SyntheticResearchClient(DATA)

    results = client.search(
        "high severity incident routing human approval"
    )

    assert results
    assert results[0].url == "synthetic://SYN-001"


def test_unrelated_question_has_no_results():
    """The tool must not invent evidence for unrelated topics."""

    client = SyntheticResearchClient(DATA)

    results = client.search(
        "volcanic magma seismology"
    )

    assert results == []


def test_search_respects_result_limit():
    """The tool should honor the requested maximum."""

    client = SyntheticResearchClient(DATA)

    results = client.search(
        "incident routing",
        limit=2
    )

    assert len(results) <= 2


def test_existing_agent_can_use_synthetic_tool():
    """The CMU agent should work with our research client."""

    agent = ResearchAgent(
        ResearchConfig(
            max_queries=2,
            max_sources=3
        )
    )

    agent.web = SyntheticResearchClient(DATA)
    agent.llm_writer = OfflineWriter()

    report = agent.research(
        "high severity incident routing human approval"
    )

    # The report must contain references.
    assert report.references

    # At least one source must mention human approval.
    assert any(
        "human approval" in source.content.lower()
        for source in report.references
    )


Writing tests/test_research_tools.py


In [19]:

# PHASE 1A - AUTOMATED TESTS

%cd /content/drive/MyDrive/InsightForge_CMU/insightforge

# Run the original CMU tests plus our new tests.
!python -m pytest tests -q


/content
......                                                                   [100%]
6 passed in 1.38s


In [20]:

# PHASE 1A - PREPARE FILES FOR GITHUB

from pathlib import Path
from zipfile import ZipFile
from google.colab import files

PROJECT = Path(
    "/content/drive/MyDrive/InsightForge_CMU/insightforge"
)

# Only the three new files we created in Phase 1A.
new_files = [
    "research_agent/research_tools.py",
    "data/synthetic/research_notes.json",
    "tests/test_research_tools.py"
]

zip_path = Path("/content/InsightForge_Phase1A_Update.zip")

with ZipFile(zip_path, "w") as archive:
    for relative_path in new_files:
        source = PROJECT / relative_path

        if not source.is_file():
            raise FileNotFoundError(source)

        archive.write(
            source,
            arcname=relative_path
        )

print("ZIP created successfully!")
print("Files included:", new_files)

# Download the ZIP to your computer.
files.download(str(zip_path))


ZIP created successfully!
Files included: ['research_agent/research_tools.py', 'data/synthetic/research_notes.json', 'tests/test_research_tools.py']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>